# Competency Question Evaluation

This notebook provides executable SPARQL queries demonstrating the competency questions supported by the Organic Flow Battery Ontology (OFBO).

The queries are evaluated through **Ontop**, which exposes OFBO as a virtual RDF graph. Ontop uses the OFBO ontology (`OFBO.ttl`) and OBDA mappings (`OFBO.obda`) to translate SPARQL queries into SQL queries over the underlying H2 database. Therefore, RDF materialization is not required.

Each competency question is represented by a separate SPARQL query file (`.rq`). For each query, this notebook:

1. states the competency question;
2. displays the SPARQL query;
3. executes it against the Ontop endpoint
4. presents the returned bindings as a table.

## Requirements

- Ontop must be running and configured with `OFBO.ttl`, `OFBO.obda`, and the H2 database.
- Default Ontop endpoint: `http://localhost:8080/sparql`
- Python dependencies: `requests` and `pandas`.

In [ ]:
from pathlib import Path
from io import StringIO
from IPython.display import Markdown, display

import pandas as pd
import requests


ONTOP_ENDPOINT = "http://localhost:8080/sparql"
TIMEOUT_SECONDS = 120


def run_cq(query_file, endpoint=ONTOP_ENDPOINT):
    query = Path(query_file).read_text(encoding="utf-8")

    response = requests.post(
        endpoint,
        data={"query": query},
        headers={"Accept": "text/csv"},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    results = pd.read_csv(StringIO(response.text))
    return query, results

## CQ1 — Which molecules share the same molecular scaffold?

This competency question is addressed using the `ofbo:OFBO_0009101` (*has core SMILES*) data property. Each molecule is linked to a Core SMILES representation of its structural scaffold through the OBDA mappings.

The query identifies pairs of distinct molecules that have the same Core SMILES value. For demonstration purposes, the result set is restricted to ten molecule pairs. The query returns the molecule identifiers, their IUPAC names where available, and the shared scaffold representation.

This query demonstrates that molecular records from the mapped data sources can be compared on the basis of their common structural scaffold.

In [7]:
query, results = run_cq("queries/cq1_same_scaffold.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo: <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>

SELECT DISTINCT
    ?molecule1
    ?name1
    ?molecule2
    ?name2
    ?coreSMILES
WHERE {
    ?molecule1 a ofbo:OFBO_0000012 ;
               ofbo:OFBO_0009101 ?coreSMILES .

    ?molecule2 a ofbo:OFBO_0000012 ;
               ofbo:OFBO_0009101 ?coreSMILES .

    OPTIONAL { ?molecule1 schema:iupacName ?name1 . }
    OPTIONAL { ?molecule2 schema:iupacName ?name2 . }

    # Retain one ordering of each pair and exclude self-matches.
    FILTER(STR(?molecule1) < STR(?molecule2))
}
ORDER BY ?coreSMILES ?molecule1 ?molecule2
LIMIT 10
```

### Results

,molecule1,name1,molecule2,name2,coreSMILES
0,https://w3id.org/ofbo#C4SC03030C_85,4-hydroxy-10-iminoanthracen-9-one,https://w3id.org/ofbo#MOL_1086412_125,4-hydroxy-10-iminoanthracen-9-one,N=C1c2ccccc2C(=O)c2ccccc21
1,https://w3id.org/ofbo#jbbabio_333,"phenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_334,"1-methyl-7-propan-2-ylphenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
2,https://w3id.org/ofbo#jbbabio_333,"phenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_335,"2,7-dihydroxyphenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
3,https://w3id.org/ofbo#jbbabio_333,"phenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_9,"phenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
4,https://w3id.org/ofbo#jbbabio_334,"1-methyl-7-propan-2-ylphenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_335,"2,7-dihydroxyphenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
5,https://w3id.org/ofbo#jbbabio_334,"1-methyl-7-propan-2-ylphenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_9,"phenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
6,https://w3id.org/ofbo#jbbabio_335,"2,7-dihydroxyphenanthrene-9,10-dione",https://w3id.org/ofbo#jbbabio_9,"phenanthrene-9,10-dione",O=C1C(=O)c2ccccc2-c2ccccc21
7,https://w3id.org/ofbo#jbbabio_336,"4-(3,5-dimethyl-4-oxocyclohexa-2,5-dien-1-ylid...",https://w3id.org/ofbo#jbbabio_337,"4-[4-oxo-3,5-di(propan-2-yl)cyclohexa-2,5-dien...",O=C1C=CC(=C2C=CC(=O)C=C2)C=C1
8,https://w3id.org/ofbo#jbbabio_336,"4-(3,5-dimethyl-4-oxocyclohexa-2,5-dien-1-ylid...",https://w3id.org/ofbo#jbbabio_338,"2,6-dibutyl-4-(3,5-dibutyl-4-oxocyclohexa-2,5-...",O=C1C=CC(=C2C=CC(=O)C=C2)C=C1
9,https://w3id.org/ofbo#jbbabio_337,"4-[4-oxo-3,5-di(propan-2-yl)cyclohexa-2,5-dien...",https://w3id.org/ofbo#jbbabio_338,"2,6-dibutyl-4-(3,5-dibutyl-4-oxocyclohexa-2,5-...",O=C1C=CC(=C2C=CC(=O)C=C2)C=C1


## CQ2 — Which molecules have reported redox potentials under acidic, neutral, and basic conditions?

This query retrieves experimentally reported redox potentials together with the pH of the chemical system used for the measurement.

For demonstration, the query is restricted to the mapped dataset from
[10.1038/srep39101](https://doi.org/10.1038/srep39101), which contains
measurements at pH 0, pH 7, and pH 13. These values represent acidic,
neutral, and basic conditions, respectively.

Each result row links a molecule to:

- its IUPAC name;
- the reported redox potential;
- the potential reference electrode;
- the pH of the experimental chemical system; and
- the chemical composition of that system.

The returned measurements demonstrate that OFBO can represent and query
redox potentials in relation to experimental pH conditions.

In [11]:
query, results = run_cq("queries/cq2_redox_potential_ph_conditions.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?molecule
    ?name
    ?pH
    ?electrolyte
    ?redoxPotential
    ?reference
WHERE {
    # Molecule and its associated electrochemical reaction
    ?molecule a ofbo:OFBO_0000012 ;
              ofbo:OFBO_0009206 ?reaction ;
              schema:iupacName ?name .

    # Redox potential associated with that reaction
    ?potential a ofbo:OFBO_0001104 ;
               sio:SIO_000011 ?reaction ;
               qudt:numericValue ?redoxPotential ;
               ofbo:OFBO_0009202 ?reference ;
               prov:wasGeneratedBy ?experiment ;
               prov:hadPrimarySource <https://doi.org/10.1038/srep39101> .

    # Experimental system used in the measurement
    ?experiment prov:used ?system .

    ?system schema:chemicalComposition ?electrolyte ;
            sio:SIO_000223 ?pHProperty .

    ?pHProperty a ofbo:OFBO_0001016 ;
                qudt:numericValue ?pH .
}
ORDER BY ?name ?pH
LIMIT 15
```

### Results

,molecule,name,pH,electrolyte,redoxPotential,reference
0,https://w3id.org/ofbo#1038srep39101_19,"1,2,5,8-tetrahydroxyanthracene-9,10-dione",0,1 M H2SO4,-0.0845,SHE
1,https://w3id.org/ofbo#1038srep39101_19,"1,2,5,8-tetrahydroxyanthracene-9,10-dione",7,1 M KCl,-0.4143,SHE
2,https://w3id.org/ofbo#1038srep39101_19,"1,2,5,8-tetrahydroxyanthracene-9,10-dione",13,1 M KOH,-0.6910,SHE
3,https://w3id.org/ofbo#1038srep39101_14,"1,2-dihydroxyanthracene-9,10-dione",0,1 M H2SO4,0.0054,SHE
4,https://w3id.org/ofbo#1038srep39101_14,"1,2-dihydroxyanthracene-9,10-dione",7,1 M KCl,-0.4006,SHE
5,https://w3id.org/ofbo#1038srep39101_14,"1,2-dihydroxyanthracene-9,10-dione",13,1 M KOH,-0.5462,SHE
6,https://w3id.org/ofbo#1038srep39101_18,"1,3,8-trihydroxy-6-methylanthracene-9,10-dione",0,1 M H2SO4,-0.0410,SHE
7,https://w3id.org/ofbo#1038srep39101_18,"1,3,8-trihydroxy-6-methylanthracene-9,10-dione",7,1 M KCl,-0.3862,SHE
8,https://w3id.org/ofbo#1038srep39101_18,"1,3,8-trihydroxy-6-methylanthracene-9,10-dione",13,1 M KOH,-0.6310,SHE
9,https://w3id.org/ofbo#1038srep39101_20,"1,4,5,8-tetrahydroxyanthracene-9,10-dione",0,1 M H2SO4,0.2004,SHE


## CQ3 — Which computational methods were used to calculate a given property for a molecule?

This query retrieves calculated redox potentials together with the computational method used to generate them. It follows the provenance chain from a molecule, to its associated electrochemical reaction, to a redox-potential value, and finally to the computational process that generated that value.

The query is not restricted to a specific source dataset. For demonstration purposes, it returns at most ten results.

The reported method corresponds to the OFBO property `ofbo:OFBO_0009102` (*has level of theory*). Examples in the mapped data include B3LYP, PBE, DFT (PCM), DFT (SM8), QM/MM, and HF-3c.

In [17]:
query, results = run_cq("queries/cq3_computational_methods.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?smiles
    ?redoxPotential
    ?reference
    ?method
WHERE {
    ?molecule a ofbo:OFBO_0000012 ;
              schema:smiles ?smiles ;
              ofbo:OFBO_0009206 ?reaction .

    ?potential a ofbo:OFBO_0001104 ;
               sio:SIO_000011 ?reaction ;
               qudt:numericValue ?redoxPotential ;
               ofbo:OFBO_0009202 ?reference ;
               prov:wasGeneratedBy ?calculation .

    ?calculation a ofbo:OFBO_0008000 ;
                 ofbo:OFBO_0009102 ?method .
}
ORDER BY ?smiles ?method
LIMIT 10
```

### Results

,smiles,redoxPotential,reference,method
0,C=CC1=c2c(C=C)c(C=C)c(C=C)c(C=C)c2=C(C=C)C(=O)...,0.950,SHE,PBE
1,C=CC1=c2c(C=C)c3c(C=C)c(C=C)c(C=C)c(C=C)c3c(C=...,0.900,SHE,PBE
2,C=CC1=c2cc3ccccc3cc2=CC(=O)C1=O,1.130,SHE,PBE
3,C=CC1=c2ccccc2=CC(=O)C1=O,1.050,SHE,PBE
4,C=CCOc1ccc(O)c2c1C(=O)c1ccccc1C2=O,0.131,SHE,DFT (PCM)
5,C=CCOc1ccc(O)c2c1C(=O)c1ccccc1C2=O,0.057,SHE,DFT (SM8)
6,C=CCOc1ccc(OC)c2c1C(=O)c1ccccc1C2=O,0.318,SHE,DFT (PCM)
7,C=CCOc1ccc(OC)c2c1C(=O)c1ccccc1C2=O,0.325,SHE,DFT (SM8)
8,C=CCOc1ccc(OC)c2c1C(=O)c1ccccc1C2=O,0.164,SHE,QM/MM
9,C=CCOc1ccc(OCC=C)c2c1C(=O)c1ccccc1C2=O,0.321,SHE,DFT (PCM)


## CQ4 — Which molecules undergo degradation pathways?

This query demonstrates the representation of molecular degradation pathways in OFBO using bromination as an example. Bromination is currently the degradation pathway instantiated in the mapped data.

The query retrieves the SMILES representation of the parent molecule, its brominated product, the reported stability comment, calculated reaction-energy values, and the computational method used for the calculation.

Although the ontology also defines anthrone formation, dimerization, and gem-diol formation, corresponding instances are not currently included in the OBDA mappings. Therefore, this competency-question demonstration uses bromination.

In [20]:
query, results = run_cq("queries/cq4_degradation_pathways.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>
PREFIX rxno:   <http://purl.obolibrary.org/obo/RXNO_>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT
    ?smiles
    ?brominatedProduct
    ?stabilityComment
    ?reactionEnergy
    ?method
WHERE {
    ?degradationReaction a ofbo:OFBO_0007202 ;
                          ofbo:OFBO_0009207 ?molecule ;
                          rxno:0000423 ?product ;
                          sio:SIO_000223 ?energy .

    ?molecule schema:smiles ?smiles .

    ?product schema:smiles ?brominatedProduct .

    ?energy a ofbo:OFBO_0001230 ;
            qudt:numericValue ?reactionEnergy ;
            prov:wasGeneratedBy ?calculation .

    ?calculation ofbo:OFBO_0009102 ?method .

    OPTIONAL {
        ?degradationReaction rdfs:comment ?stabilityComment .
    }
}
ORDER BY ?smiles ?reactionEnergy
LIMIT 10
```

### Results

,smiles,brominatedProduct,stabilityComment,reactionEnergy,method
0,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,O=C1c2cc(Br)c(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(...,Poor,-77,B3LYP/6-31G*
1,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,Oc1c2cc(Br)c(S(=O)(=O)O)c(O)c2c(O)c2c1ccc(S(=O...,Good,-65,B3LYP/6-31G*
2,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,Oc1c2cc(Br)c(S(=O)(=O)O)c(O)c2c(O)c2c1ccc(S(=O...,Good,16,B3LYP/6-31G*
3,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,O=C1c2cc(Br)c(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(...,Poor,48,B3LYP/6-31G*
4,O=C1c2ccc(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O)O)c...,O=C1c2cc(Br)c(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O...,Good,0,B3LYP/6-31G*
5,O=C1c2ccc(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O)O)c...,Oc1c2cc(Br)c(S(=O)(=O)O)cc2c(O)c2cc(S(=O)(=O)O...,Good,0,B3LYP/6-31G*
6,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,Oc1c2c(Br)cccc2c(O)c2c(O)c(CS(=O)(=O)O)c(CS(=O...,Poor,-104,B3LYP/6-31G*
7,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,O=C1c2c(Br)cccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(...,Apparently stable,13,B3LYP/6-31G*
8,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,Oc1c2c(Br)cccc2c(O)c2c(O)c(CS(=O)(=O)O)c(CS(=O...,Poor,38,B3LYP/6-31G*
9,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,O=C1c2c(Br)cccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(...,Apparently stable,74,B3LYP/6-31G*


## CQ5 — Which solvent or electrolyte system is suitable for a given redox-active material?

This query retrieves reported solubility measurements for the molecule with the following SMILES representation:

`CC(C)=CCC1=C(O)c2ccccc2C(=O)C1=O`

For each available solubility measurement, the query returns the chemical system used, its pH where reported, and the measured solubility. These results allow the available solvent/electrolyte systems to be compared for this specific redox-active material.

In the current OFBO mapping, suitability is represented through reported solubility measurements rather than through an explicit suitability assertion.

In [22]:
query, results = run_cq("queries/cq5_solvent_suitability.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?solventSystem
    ?pH
    ?solubility
    ?unit
WHERE {
    ?molecule schema:smiles "CC(C)=CCC1=C(O)c2ccccc2C(=O)C1=O" ;
              sio:SIO_000223 ?solubilityProperty .

    ?solubilityProperty a ofbo:OFBO_0002008 ;
                         qudt:numericValue ?solubility ;
                         qudt:unit ?unit ;
                         prov:wasGeneratedBy ?measurement .

    ?measurement prov:used ?system .

    ?system schema:chemicalComposition ?solventSystem .

    OPTIONAL {
        ?system sio:SIO_000223 ?pHProperty .

        ?pHProperty a ofbo:OFBO_0001016 ;
                    qudt:numericValue ?pH .
    }
}
ORDER BY ?pH ?solventSystem
```

### Results

,solventSystem,pH,solubility,unit
0,1 M H2SO4,0,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L
1,1 M KCl,7,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L
2,1 M KOH,13,0.665,http://qudt.org/vocab/unit/MOL-PER-L


## CQ6 — Which molecules are compatible with brominated electrolytes?

This query retrieves molecules for which bromination reactions and associated stability assessments are reported. Bromination is represented in OFBO as a degradation reaction (`ofbo:OFBO_0007202`).

The query returns the SMILES representation of the parent molecule, the brominated product, and the reported stability assessment. These data can be used to identify molecules evaluated for compatibility with brominated electrolyte environments.

In the current mapping, compatibility is represented as an assessment of stability against bromination, rather than as a separate Boolean compatibility assertion.

In [23]:
query, results = run_cq("queries/cq6_brominated_electrolyte_compatibility.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX rxno:   <http://purl.obolibrary.org/obo/RXNO_>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT
    ?smiles
    ?brominatedProduct
    ?stabilityAssessment
WHERE {
    ?brominationReaction a ofbo:OFBO_0007202 ;
                          ofbo:OFBO_0009207 ?molecule ;
                          rdfs:comment ?stabilityAssessment .

    ?molecule schema:smiles ?smiles .

    OPTIONAL {
        ?brominationReaction rxno:0000423 ?product .
        ?product schema:smiles ?brominatedProduct .
    }
}
ORDER BY ?smiles
LIMIT 10
```

### Results

,smiles,brominatedProduct,stabilityAssessment
0,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,O=C1c2cc(Br)c(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(...,Poor
1,O=C1c2ccc(S(=O)(=O)O)c(O)c2C(=O)c2c1ccc(S(=O)(...,Oc1c2cc(Br)c(S(=O)(=O)O)c(O)c2c(O)c2c1ccc(S(=O...,Good
2,O=C1c2ccc(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O)O)c...,O=C1c2cc(Br)c(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O...,Good
3,O=C1c2ccc(S(=O)(=O)O)cc2C(=O)c2cc(S(=O)(=O)O)c...,Oc1c2cc(Br)c(S(=O)(=O)O)cc2c(O)c2cc(S(=O)(=O)O...,Good
4,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,O=C1c2c(Br)cccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(...,Apparently stable
5,O=C1c2ccccc2C(=O)c2c(O)c(CS(=O)(=O)O)c(CS(=O)(...,Oc1c2c(Br)cccc2c(O)c2c(O)c(CS(=O)(=O)O)c(CS(=O...,Poor
6,O=C1c2ccccc2C(=O)c2c1cc(S(=O)(=O)O)c(O)c2O,O=C1c2cc(Br)ccc2C(=O)c2c1cc(S(=O)(=O)O)c(O)c2O,Poor
7,O=C1c2ccccc2C(=O)c2c1cc(S(=O)(=O)O)c(O)c2O,Oc1c2cc(Br)ccc2c(O)c2c1cc(S(=O)(=O)O)c(O)c2O,Good
8,O=C1c2ccccc2C(=O)c2cc(S(=O)(=O)O)ccc21,O=C1c2cc(Br)ccc2C(=O)c2cc(S(=O)(=O)O)ccc21,Good
9,O=C1c2ccccc2C(=O)c2cc(S(=O)(=O)O)ccc21,Oc1c2cc(Br)ccc2c(O)c2cc(S(=O)(=O)O)ccc21,Good


## CQ7 — For a given molecule, which predicted molecular properties have not been experimentally verified?

This query retrieves machine-learning prediction values that are not linked to an experimental process in the current OFBO knowledge graph.

A value is considered predicted when it was generated by an instance of `ofbo:OFBO_0008002` (*machine learning process*). The query excludes any prediction value that is also generated by an instance of `ofbo:OFBO_0008500` (*experimental process*).

The result therefore lists molecular predictions for which no experimental provenance is attached to the same value entity.

In [26]:
query, results = run_cq("queries/cq7_unverified_predictions.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Results"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?smiles
    ?predictedValue
    ?unit
WHERE {
    ?molecule schema:smiles ?smiles ;
              <http://semanticscience.org/resource/SIO_000223> ?prediction .

    ?prediction qudt:numericValue ?predictedValue ;
                qudt:unit ?unit ;
                prov:wasGeneratedBy ?mlProcess .

    FILTER NOT EXISTS {
        ?prediction prov:wasGeneratedBy ?experimentalProcess .
        ?experimentalProcess a ofbo:OFBO_0008500 .
    }
}
LIMIT 10
```

### Results

,smiles,predictedValue,unit
0,O=C1C=CC2=C(C(F)(F)F)C(=O)C=CC2=C1,-9.110071,http://qudt.org/vocab/unit/KiloJ-PER-MOL
1,O=C1C=CC2=C(C(F)(F)F)C(=O)C=CC2=C1,6.034000,http://qudt.org/vocab/unit/EV
2,O=C1C=CC2=C(C(F)(F)F)C(=O)C=CC2=C1,-9.092480,http://qudt.org/vocab/unit/KiloJ-PER-MOL
3,O=C1C=CC2=C(C(F)(F)F)C(=O)C=CC2=C1,6.159041,http://qudt.org/vocab/unit/EV
4,O=C1C=CC2=C(C(F)(F)F)C(=O)C=CC2=C1,-36.970000,http://qudt.org/vocab/unit/KiloJ-PER-MOL
5,N#Cc1cccc2cc3c(cc12)=CC(=O)C(=O)C=3,-97.330000,http://qudt.org/vocab/unit/KiloJ-PER-MOL
6,N#Cc1cccc2cc3c(cc12)=CC(=O)C(=O)C=3,-13.470071,http://qudt.org/vocab/unit/KiloJ-PER-MOL
7,N#Cc1cccc2cc3c(cc12)=CC(=O)C(=O)C=3,-13.491973,http://qudt.org/vocab/unit/KiloJ-PER-MOL
8,N#Cc1cccc2cc3c(cc12)=CC(=O)C(=O)C=3,5.984000,http://qudt.org/vocab/unit/EV
9,N#Cc1cccc2cc3c(cc12)=CC(=O)C(=O)C=3,5.888134,http://qudt.org/vocab/unit/EV


## CQ8 — Which computational descriptors correlate with experimentally measured redox potentials?

This competency question requires two steps:

1. SPARQL retrieves paired experimental and computational values for the same molecule.
2. A statistical analysis calculates the correlation between these values.

This query retrieves the SMILES representation of each molecule, its experimentally measured redox potential, and its calculated solvation free energy. The query also returns the computational method used to obtain the solvation free energy.

The results can subsequently be used to calculate a Pearson correlation coefficient. This demonstrates that OFBO can connect experimental electrochemical measurements and computational molecular descriptors through their shared molecular entity.

In [29]:
query, results = run_cq("queries/cq8_redox_potential_solvent_energy.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Retrieved paired values"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?molecule
    ?smiles
    ?experimentalRedoxPotential
    ?solvationFreeEnergy
    ?method
WHERE {
    ?molecule ofbo:OFBO_0009101 ?coreSmiles ;
              ofbo:OFBO_0009206 ?reaction ;
              sio:SIO_000223 ?solvationEnergy .

    # Prefer a full SMILES that differs from the Core SMILES.
    OPTIONAL {
        ?molecule schema:smiles ?fullSmiles .
        FILTER(?fullSmiles != ?coreSmiles)
    }

    # Where no separate full SMILES exists, use the Core SMILES.
    BIND(COALESCE(?fullSmiles, ?coreSmiles) AS ?smiles)

    ?redoxPotential a ofbo:OFBO_0001104 ;
                    sio:SIO_000011 ?reaction ;
                    qudt:numericValue ?experimentalRedoxPotential ;
                    prov:wasGeneratedBy ?experiment .

    ?experiment a ofbo:OFBO_0008500 .

    ?solvationEnergy a ofbo:OFBO_0002005 ;
                     qudt:numericValue ?solvationFreeEnergy ;
                     prov:wasGeneratedBy ?calculation .

    ?calculation ofbo:OFBO_0009102 ?method .
}
ORDER BY ?molecule
LIMIT 100
```

### Retrieved paired values

,molecule,smiles,experimentalRedoxPotential,solvationFreeEnergy,method
0,https://w3id.org/ofbo#MOL_s41560_018_0167_3_1,c1ccc2nc3ccccc3nc2c1,-0.36,11.163315,B3LYP-D3
1,https://w3id.org/ofbo#MOL_s41560_018_0167_3_2,O=C1C=c2nc3ccccc3nc2=CC1=O,-0.82,-480.746161,B3LYP-D3
2,https://w3id.org/ofbo#MOL_s41560_018_0167_3_4,O=C1C=c2nc3ccc(S(=O)(=O)O)cc3nc2=CC1=O,-0.86,-957.015418,B3LYP-D3
3,https://w3id.org/ofbo#MOL_s41560_018_0167_3_6,O=C1C=c2nc3ccc(C(=O)O)cc3nc2=CC1=O,-0.88,-975.280028,B3LYP-D3


In [30]:
correlation = results[
    ["experimentalRedoxPotential", "solvationFreeEnergy"]
].corr(method="pearson").iloc[0, 1]

print(f"Pearson correlation coefficient: {correlation:.3f}")

Pearson correlation coefficient: 0.915


## CQ9 — Which redox-active molecules are promising candidates for high energy density?

A complete battery-energy-density calculation is outside the current OFBO scope because it requires a paired positive and negative electrolyte, cell voltage, state of charge, and battery-level parameters.

However, OFBO can support molecular-level candidate screening by retrieving two relevant experimentally measured properties under the same chemical system:

- redox potential, which contributes to the attainable cell voltage; and
- solubility, which limits the concentration of active material.

The following SPARQL query retrieves Core SMILES, chemical-system composition, redox potential, and solubility for the same molecule and experimental system. A simple Python post-processing step then identifies Pareto-optimal candidates that maximise both reported solubility and redox potential.

This is a two-property screening demonstration, not a calculation of complete battery energy density.

In [31]:
query, results = run_cq("queries/cq9_redox_potential_solubility.rq")

display(Markdown("### SPARQL query"))
display(Markdown(f"```sparql\n{query}\n```"))

display(Markdown("### Retrieved experimental measurements"))
results

### SPARQL query

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>

SELECT DISTINCT
    ?molecule
    ?smiles
    ?solventSystem
    ?pH
    ?redoxPotential
    ?solubility
    ?solubilityUnit
    ?reference
WHERE {
    # Core SMILES is used to avoid duplicate values inferred through
    # ofbo:OFBO_0009101 rdfs:subPropertyOf schema:smiles.
    ?molecule a ofbo:OFBO_0000012 ;
              ofbo:OFBO_0009101 ?smiles ;
              ofbo:OFBO_0009206 ?reaction ;
              sio:SIO_000223 ?solubilityProperty .

    # Experimentally measured solubility
    ?solubilityProperty a ofbo:OFBO_0002008 ;
                         qudt:numericValue ?solubility ;
                         qudt:unit ?solubilityUnit ;
                         prov:wasGeneratedBy ?solubilityMeasurement .

    ?solubilityMeasurement a ofbo:OFBO_0008500 ;
                           prov:used ?system .

    # Experimental redox potential measured in the same system
    ?redoxProperty a ofbo:OFBO_0001104 ;
                   sio:SIO_000011 ?reaction ;
                   qudt:numericValue ?redoxPotential ;
                   ofbo:OFBO_0009202 ?reference ;
                   prov:wasGeneratedBy ?redoxMeasurement .

    ?redoxMeasurement a ofbo:OFBO_0008500 ;
                      prov:used ?system .

    # Chemical-system information
    ?system schema:chemicalComposition ?solventSystem .

    OPTIONAL {
        ?system sio:SIO_000223 ?pHProperty .
        ?pHProperty a ofbo:OFBO_0001016 ;
                    qudt:numericValue ?pH .
    }
}
ORDER BY DESC(?solubility) DESC(?redoxPotential)
```

### Retrieved experimental measurements

,molecule,smiles,solventSystem,pH,redoxPotential,solubility,solubilityUnit,reference
0,https://w3id.org/ofbo#202312608_1,c1ccc2c(c1)Nc1ccccc1S2,3 M H2SO4,0,0.6200,<0.001,http://qudt.org/vocab/unit/MOL-PER-L,SHE
1,https://w3id.org/ofbo#1038srep39101_27,Oc1ccc(O)cc1,1 M H2SO4,0,0.7010,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE
2,https://w3id.org/ofbo#1038srep39101_10,O=C1C=CC(=O)c2ccccc21,1 M KOH,13,0.4000,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE
3,https://w3id.org/ofbo#1038srep39101_26,O=C1C=CC(=O)C=C1,1 M H2SO4,0,0.3941,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE
4,https://w3id.org/ofbo#1038srep39101_25,O=C1C=CC(=O)C=C1,1 M H2SO4,0,0.3824,< 0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE
...,...,...,...,...,...,...,...,...
80,https://w3id.org/ofbo#1038srep39101_5,O=C1c2ccccc2C(=O)c2ccccc21,1 M H2SO4,0,0.0154,0.019,http://qudt.org/vocab/unit/MOL-PER-L,SHE
81,https://w3id.org/ofbo#1038srep39101_3,O=C1c2ccccc2C(=O)c2ccccc21,1 M KCl,7,-0.2327,0.016,http://qudt.org/vocab/unit/MOL-PER-L,SHE
82,https://w3id.org/ofbo#1038srep39101_27,Oc1ccc(O)cc1,1 M KCl,7,0.2771,0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE
83,https://w3id.org/ofbo#1038srep39101_7,O=C1C=Cc2ccccc2C1=O,1 M KCl,7,0.2450,0.01,http://qudt.org/vocab/unit/MOL-PER-L,SHE


In [32]:
screening = results.copy()

screening["redoxPotential"] = pd.to_numeric(
    screening["redoxPotential"],
    errors="coerce"
)

screening["solubility"] = pd.to_numeric(
    screening["solubility"],
    errors="coerce"
)

screening = screening.dropna(
    subset=["redoxPotential", "solubility"]
)

# This demonstration compares only values reported in mol L^-1.
screening = screening[
    screening["solubilityUnit"].str.contains(
        "MOL-PER-L",
        na=False
    )
].copy()


def is_pareto_optimal(data):
    """Return True for rows not dominated in both screening objectives."""
    optimal = []

    for _, candidate in data.iterrows():
        dominated = (
            (data["solubility"] >= candidate["solubility"])
            & (data["redoxPotential"] >= candidate["redoxPotential"])
            & (
                (data["solubility"] > candidate["solubility"])
                | (data["redoxPotential"] > candidate["redoxPotential"])
            )
        ).any()

        optimal.append(not dominated)

    return optimal


screening["paretoOptimal"] = is_pareto_optimal(screening)

screening = screening.sort_values(
    by=["paretoOptimal", "solubility", "redoxPotential"],
    ascending=[False, False, False],
)

screening[
    [
        "smiles",
        "solventSystem",
        "pH",
        "redoxPotential",
        "solubility",
        "solubilityUnit",
        "reference",
        "paretoOptimal",
    ]
]

,smiles,solventSystem,pH,redoxPotential,solubility,solubilityUnit,reference,paretoOptimal
39,Oc1ccccc1O,1 M H2SO4,0,0.78310,1.650,http://qudt.org/vocab/unit/MOL-PER-L,SHE,True
52,Oc1ccccc1O,1 M H2SO4,0,0.86300,0.675,http://qudt.org/vocab/unit/MOL-PER-L,SHE,True
40,Oc1ccccc1O,1 M KCl,7,0.46450,1.650,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
41,Oc1ccc(O)cc1,1 M KOH,13,0.11100,1.650,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
42,Oc1ccc(O)cc1,1 M KCl,7,0.23700,1.010,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
43,O=C1c2ccccc2C(=O)c2ccccc21,1 M H2SO4,0,0.16550,1.010,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
44,Oc1ccc(O)cc1,1 M KOH,13,-0.05600,1.010,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
45,O=C1C=Cc2ccccc2C1=O,1 M KOH,13,-0.48100,0.890,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
46,O=C1C=Cc2ccccc2C1=O,1 M KOH,13,-0.08140,0.865,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
47,Oc1ccc(O)cc1,1 M KOH,13,-0.07100,0.805,http://qudt.org/vocab/unit/MOL-PER-L,SHE,False
